# 03 — 连续隐变量：仍然可以精确求解
主动定义 $x\sim N(0,1)$，$y|x\sim N(wx+b,\sigma^2 I)$。$x$ 不可观测，$y$ 是二维向量。联合密度仍然只需相乘；边缘化由求和变成积分。由于线性高斯结构，边缘与后验都有解析解。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp
from scipy.stats import norm, multivariate_normal
rng = np.random.default_rng(42)
# 仅用于制造观测数据；训练代码不使用真实类别或真实参数。
centers = np.array([[-2., 0.], [2., 0.], [0., 3.]])
y = centers[rng.integers(0, 3, 900)] + 0.45*rng.normal(size=(900, 2))
def compare(samples):
    fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
    for a, data, title in zip(ax, [y, samples], ['Observed', 'Generated']):
        a.scatter(*data.T, s=5, alpha=.5); a.set_title(title); a.set_aspect('equal')
    plt.show()


In [ ]:
sigma = .45
b = y.mean(0)
# 固定 sigma 时的一维线性高斯模型 MLE（最大特征值方向）
vals, vecs = np.linalg.eigh((y-b).T @ (y-b)/len(y))
w = vecs[:,-1] * np.sqrt(max(vals[-1]-sigma**2,0))
def log_joint(x, obs):
    return norm.logpdf(x) + norm.logpdf(obs, x[...,None]*w+b, sigma).sum(-1)
model = multivariate_normal(b, np.outer(w,w)+sigma**2*np.eye(2))
v = 1/(1+w@w/sigma**2)
m = v*((y-b)@w)/sigma**2
print('w, b:', w,b, 'posterior variance:',v)
obs = y[0]
grid = np.linspace(-8,8,4001); dx=grid[1]-grid[0]
lj = log_joint(grid,obs)
log_evidence = logsumexp(lj)+np.log(dx)
print('integral:',log_evidence,'analytic:',model.logpdf(obs))
assert abs(log_evidence-model.logpdf(obs))<1e-6
plt.plot(grid,np.exp(lj-log_evidence),label='numerical'); plt.plot(grid,norm.pdf(grid,m[0],np.sqrt(v)),'--',label='analytic'); plt.legend(); plt.show()
compare(rng.normal(size=(len(y),1))*w+b+sigma*rng.normal(size=y.shape))

解析后验为 $N(m,v)$，其中 $v=(1+w^Tw/\sigma^2)^{-1}$，$m=v w^T(y-b)/\sigma^2$。改变网格密度，观察积分误差。连续隐变量不必然意味着不可解；线性高斯结构才是解析解的原因。模型只能生成一个高斯分布，因此不能表达三个峰。